In [2]:
!pip install scikit-learn

In [4]:
import sklearn
print(sklearn.__version__)

1.9.0


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_csv('Final_data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 661 entries, 0 to 660
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        661 non-null    str  
 1   company     661 non-null    str  
 2   year        661 non-null    int64
 3   Price       661 non-null    int64
 4   kms_driven  661 non-null    int64
 5   fuel_type   661 non-null    str  
dtypes: int64(3), str(3)
memory usage: 31.1 KB


In [14]:
#Divide dataset into X and y
X = df[['company', 'name','year','kms_driven', 'fuel_type']]
y = df[['Price']]

In [15]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

In [17]:
ohe = OneHotEncoder()
ohe.fit(X[['company', 'name', 'fuel_type']])
ct = make_column_transformer((OneHotEncoder(categories = ohe.categories_),['company', 'name', 'fuel_type']), remainder = 'passthrough')
model = LinearRegression()
pipe = make_pipeline(ct, model)

In [21]:
#Divide into train and test, train and calculate accuracy
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
scores = []
for i in range(0,101):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = i)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns =['Prediction'])
    result = pd.concat([y_test.reset_index(drop = True), y_pred] , axis =1)
    score = r2_score(result['Price'],result['Prediction'])
    scores.append(score)

In [22]:
#get index of max value
index = np.argmax(scores)

In [24]:
#lets split using best index and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = index)
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['company','name','year','kms_driven','fuel_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concaten

In [26]:
#Check for user input
myinput = [['Ford', 'Figo', 2020, 80000, 'Petrol']]
columns = ['company', 'name', 'year', 'kms_driven', 'fuel_type']
myinput = pd.DataFrame(data = myinput , columns = columns)
result = pipe.predict(myinput)
print("Predicted price is :",round(result[0,0]))

Predicted price is : 455403


In [27]:
#export pipe(model) using pickle or joblib
import pickle
pickle.dump(pipe, open("pipe.pkl","wb+"))